In [1]:
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import sys
import os
current_dir = os.getcwd()
project_root = os.path.dirname(current_dir)
if project_root not in sys.path:
    sys.path.append(project_root)

from src.decision_tree import DecisionTreeRegressor # Dùng lại cây của bạn
from src.random_forest import RandomForestRegressor # Thay 'your_module' bằng tên file chứa class của bạn

# Thiết lập hiển thị cho đồ thị
plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

In [2]:
train_path = os.path.join(project_root, 'data', 'train.csv')
val_path   = os.path.join(project_root, 'data', 'val.csv')
test_path  = os.path.join(project_root, 'data', 'test.csv')

df_train = pd.read_csv(train_path)
df_val   = pd.read_csv(val_path)
df_test = pd.read_csv(test_path)

In [3]:
X_train, y_train = df_train.iloc[:, :-1].values, df_train.iloc[:, -1].values
X_val, y_val = df_val.iloc[:, :-1].values, df_val.iloc[:, -1].values
X_test, y_test = df_test.iloc[:, :-1].values, df_test.iloc[:, -1].values

In [4]:
# Khởi tạo và huấn luyện mô hình cơ bản
rf_model_base = RandomForestRegressor(
    n_estimators=50, max_depth=10, min_samples_split=2, max_features=4
)
rf_model_base.fit(X_train, y_train)

# Dự đoán và đánh giá
y_pred_base = rf_model_base.predict(X_test)
mse_base = mean_squared_error(y_test, y_pred_base)
rmse_base = np.sqrt(mse_base)

print(f"Kết quả mô hình cơ bản (RMSE): {rmse_base:.4f}")

Kết quả mô hình cơ bản (RMSE): 9.5656


In [5]:
from sklearn.ensemble import RandomForestRegressor as SklearnRandomForestRegressor
from sklearn.model_selection import GridSearchCV

In [6]:
# --- 2. Định nghĩa Không gian Tham số ---
param_grid = {
    # Số lượng cây: Tăng dần
    'n_estimators': [50, 100, 200],
    
    # Độ sâu tối đa: Điều chỉnh để chống overfitting
    'max_depth': [70, 100, 120],
    
    # Số mẫu tối thiểu để chia: Kiểm soát kích thước nhánh
    'min_samples_split': [2, 5],
    
    # Tỷ lệ đặc trưng ngẫu nhiên: Giá trị float (tỷ lệ)
    'max_features': [0.7, 0.8] 
}

# --- 3. Cấu hình GridSearchCV ---
# Khởi tạo mô hình Random Forest của sklearn
rf_estimator = SklearnRandomForestRegressor(random_state=42)

# Khởi tạo Grid Search:
# cv=5: K-fold cross-validation 5 lần
# scoring='neg_mean_squared_error': Mục tiêu là giảm lỗi (tối ưu hóa RMSE/MSE)
grid_search = GridSearchCV(
    estimator=rf_estimator,
    param_grid=param_grid,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1, # Sử dụng tất cả các lõi CPU để tăng tốc
    verbose=1
)

print("\nBắt đầu chạy Grid Search trên mô hình Sklearn...")
grid_search.fit(X_train, y_train)


Bắt đầu chạy Grid Search trên mô hình Sklearn...
Fitting 5 folds for each of 36 candidates, totalling 180 fits


KeyboardInterrupt: 

In [7]:
# --- 4. Trích xuất kết quả ---
best_params_sklearn = grid_search.best_params_
best_rmse_sklearn = np.sqrt(-grid_search.best_score_) # Chuyển điểm âm MSE về RMSE dương

print("\n🔥 Kết quả Tinh chỉnh (Hyperparameter Tuning) từ Sklearn:")
print(best_params_sklearn)
print(f"RMSE tốt nhất tìm được (trên cross-validation): {best_rmse_sklearn:.4f}")

# --- 5. Chuyển đổi và Áp dụng cho Mô hình của bạn ---

# Lớp RandomForestRegressor tự xây dựng của bạn nhận max_features là số lượng (int).
n_features_total = X_train.shape[1]
best_max_features_ratio = best_params_sklearn['max_features']

# Chuyển đổi tỷ lệ (float) sang số lượng (int)
tuned_max_features_count = int(best_max_features_ratio * n_features_total)

# 1. Trích xuất các tham số đã tinh chỉnh
tuned_n_estimators = best_params_sklearn['n_estimators']
tuned_max_depth = best_params_sklearn['max_depth']
tuned_min_samples_split = best_params_sklearn['min_samples_split']


print("\n--- Tham số đã sẵn sàng cho mô hình tự xây dựng của bạn ---")
print(f"n_estimators: {tuned_n_estimators}")
print(f"max_depth: {tuned_max_depth}")
print(f"min_samples_split: {tuned_min_samples_split}")
print(f"max_features (Số lượng): {tuned_max_features_count} (Tương ứng với tỷ lệ {best_max_features_ratio})")


🔥 Kết quả Tinh chỉnh (Hyperparameter Tuning) từ Sklearn:
{'max_depth': 70, 'max_features': 0.8, 'min_samples_split': 2, 'n_estimators': 200}
RMSE tốt nhất tìm được (trên cross-validation): 3.2048

--- Tham số đã sẵn sàng cho mô hình tự xây dựng của bạn ---
n_estimators: 200
max_depth: 70
min_samples_split: 2
max_features (Số lượng): 5 (Tương ứng với tỷ lệ 0.8)


In [7]:

rf_model_tuned = RandomForestRegressor(
    n_estimators=200,
    max_depth=70,
    min_samples_split=2,
    max_features=5 # Dùng số lượng đã chuyển đổi
)

rf_model_tuned.fit(X_train, y_train)
joblib.dump(rf_model_tuned, 'models/scratch_randomforest.pkl')
# Dự đoán và đánh giá
y_pred_base = rf_model_tuned.predict(X_test)
mse_base = mean_squared_error(y_test, y_pred_base)
rmse_base = np.sqrt(mse_base)
r2_base = r2_score(y_test, y_pred_base)
mae_base = mean_absolute_error(y_test, y_pred_base)

In [8]:
print(rmse_base)
print(r2_base)
print(mae_base)

5.843828731376319
0.9915381321276231
4.127769521825396
